# 02 - Mô hình Hybrid KNN (Item-Based + User Preferences)

**Mục tiêu:**
Xây dựng mô hình Gợi ý dựa trên sự kết hợp giữa:
1. **Item-Based KNN:** Tính toán sự tương đồng giữa các bài hát dựa trên lịch sử nghe nhạc của người dùng (Collaborative Filtering).
2. **User Preferences:** Lọc và tăng trọng số cho các bài hát phù hợp với thể loại (Genre) và ngôn ngữ (Language) yêu thích nhất của người dùng.
Mô hình sẽ được xuất ra file `.pkl` để có thể sử dụng trực tiếp trong ứng dụng thực tế.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

# Tạo thư mục lưu model nếu chưa có
os.makedirs('models/intermediate', exist_ok=True)

print('✅ Import thư viện thành công!')


✅ Import thư viện thành công!


## 1. Tải và Lọc Dữ liệu
Sử dụng file `merged_data.csv`. Để tối ưu hóa hiệu năng mô hình Item-based KNN và giảm dung lượng model, chúng ta sẽ lọc bỏ những bài hát "đuôi dài" (long-tail) - tức là những bài hát có ít hơn 10 lượt nghe.


In [2]:
# Đọc dữ liệu
df = pd.read_csv('processed_data/merged_data.csv')
print(f'Kích thước dữ liệu gốc: {df.shape[0]:,} dòng x {df.shape[1]} cột')

# Tính số lượt nghe của mỗi bài hát
song_counts = df['song_id'].value_counts()

# Lọc các bài hát có ít nhất 10 lượt nghe
popular_songs = song_counts[song_counts >= 10].index
df_filtered = df[df['song_id'].isin(popular_songs)].reset_index(drop=True)

print(f'Số lượng bài hát ban đầu: {len(song_counts):,}')
print(f'Số lượng bài hát sau khi lọc (>=10 lượt nghe): {len(popular_songs):,}')
print(f'Kích thước dữ liệu sau khi lọc: {df_filtered.shape[0]:,} dòng')

# Tạo cột primary_genre từ genre_ids
if 'primary_genre' not in df_filtered.columns:
    df_filtered['primary_genre'] = df_filtered['genre_ids'].astype(str).str.split('|').str[0]

# Trích xuất thông tin bài hát
song_info_cols = ['song_id', 'song_name', 'artist_name', 'primary_genre', 'language']
song_details = df_filtered[song_info_cols].drop_duplicates(subset=['song_id']).set_index('song_id')

# ========== PHẦN XỬ LÝ USER PERSONAL - ĐÃ SỬA LỖI ==========
# Lấy thông tin cá nhân user
user_personal = df_filtered[['msno', 'bd', 'city', 'gender']].drop_duplicates(subset=['msno'])

# XÓA BỎ CÁC DÒNG CÓ GIÁ TRỊ NaN (TRỐNG)
print(f"Số user trước khi xóa NaN: {len(user_personal)}")
user_personal = user_personal.dropna(subset=['bd', 'city', 'gender'])
print(f"Số user sau khi xóa NaN: {len(user_personal)}")

# Mã hóa gender (chuyển male/female thành số)
from sklearn.preprocessing import LabelEncoder
le_gender = LabelEncoder()
user_personal['gender_encoded'] = le_gender.fit_transform(user_personal['gender'].astype(str))

# Chuẩn hóa các cột số
from sklearn.preprocessing import StandardScaler
scaler_personal = StandardScaler()
user_personal['bd_scaled'] = scaler_personal.fit_transform(user_personal[['bd']])
user_personal['city_scaled'] = scaler_personal.fit_transform(user_personal[['city']])
user_personal['gender_scaled'] = scaler_personal.fit_transform(user_personal[['gender_encoded']])

print(f"Đã lấy thông tin cá nhân cho {len(user_personal)} users")

# Huấn luyện mô hình tìm user tương tự
from sklearn.neighbors import NearestNeighbors

feature_cols = ['bd_scaled', 'city_scaled', 'gender_scaled']
X_personal = user_personal[feature_cols].values

user_knn = NearestNeighbors(n_neighbors=11, metric='euclidean')
user_knn.fit(X_personal)

user_to_idx_personal = {user: idx for idx, user in enumerate(user_personal['msno'])}
idx_to_user_personal = {idx: user for user, idx in user_to_idx_personal.items()}

print("✅ Đã huấn luyện mô hình tìm user tương tự")

Kích thước dữ liệu gốc: 174,010 dòng x 16 cột
Số lượng bài hát ban đầu: 20,474
Số lượng bài hát sau khi lọc (>=10 lượt nghe): 2,866
Kích thước dữ liệu sau khi lọc: 136,326 dòng
Số user trước khi xóa NaN: 10726
Số user sau khi xóa NaN: 10726
Đã lấy thông tin cá nhân cho 10726 users
✅ Đã huấn luyện mô hình tìm user tương tự


## 2. Xây dựng Hồ sơ Sở thích (User Taste Profiles)
Phân tích lịch sử của từng người dùng để tìm ra **Thể loại (Genre)** và **Ngôn ngữ (Language)** mà họ nghe nhiều nhất.


In [3]:
# Tính toán Thể loại yêu thích nhất của từng user
user_top_genre = df_filtered.groupby('msno')['primary_genre'].agg(lambda x: x.value_counts().index[0]).reset_index()
user_top_genre.columns = ['msno', 'top_genre']

# Tính toán Ngôn ngữ yêu thích nhất của từng user
user_top_lang = df_filtered.groupby('msno')['language'].agg(lambda x: x.value_counts().index[0]).reset_index()
user_top_lang.columns = ['msno', 'top_language']

# Gộp thành User Profile
user_profiles = pd.merge(user_top_genre, user_top_lang, on='msno')
user_profiles.set_index('msno', inplace=True)

print(f'Đã tạo hồ sơ sở thích cho {len(user_profiles):,} người dùng.')
user_profiles.head()


Đã tạo hồ sơ sở thích cho 10,726 người dùng.


,top_genre,top_language
msno,,
++5wYjoMgQHoRuD3GbbvmphZbBBwymzv5Q4l8sywtuU=,458,3.0
++AH7m/EQ4iKe6wSlfO/xXAJx50p+fCeTyF90GoE9Pg=,465,3.0
++e+jsxuQ8UEnmW40od9Rq3rW7+wAum4wooXyZTKJpk=,465,3.0
+/UwoUi5+rNj/F6RO6gMrMhOy0oTzs90MWKVNZs4+Wg=,465,3.0
+0e12C+p9dzDbOvKjt8eElKH9yZPshAstxjm60XFgSM=,465,3.0


## 3. Xây dựng Ma trận Tương tác Item-User
Để chạy thuật toán Item-based KNN, ta cần ma trận thưa với cấu trúc:
- Hàng (Rows): Bài hát (`song_id`)
- Cột (Columns): Người dùng (`msno`)


In [4]:
# Tạo ID số nguyên cho User và Song để đưa vào ma trận thưa
user_ids = df_filtered['msno'].astype('category').cat.categories
song_ids = df_filtered['song_id'].astype('category').cat.categories

# Dictionary ánh xạ giữa Index và ID gốc
user_to_idx = {user: i for i, user in enumerate(user_ids)}
song_to_idx = {song: i for i, song in enumerate(song_ids)}
idx_to_song = {i: song for song, i in song_to_idx.items()}

# Ánh xạ ID vào DataFrame
df_filtered['user_idx'] = df_filtered['msno'].map(user_to_idx)
df_filtered['song_idx'] = df_filtered['song_id'].map(song_to_idx)

# Trọng số tương tác = 1 (vì chỉ có target=1)
interactions = np.ones(len(df_filtered))

# Khởi tạo CSR Matrix: (Số bài hát x Số người dùng)
item_user_matrix = csr_matrix((interactions, (df_filtered['song_idx'], df_filtered['user_idx'])), 
                              shape=(len(song_ids), len(user_ids)))

print(f'Kích thước ma trận thưa Item-User: {item_user_matrix.shape}')


Kích thước ma trận thưa Item-User: (2866, 10726)


## 4. Huấn luyện Mô hình Item-based KNN
Sử dụng `NearestNeighbors` để tính khoảng cách Cosine giữa các bài hát.


In [5]:
# Khởi tạo mô hình KNN
# Sử dụng 'cosine' cho dữ liệu Sparse Matrix
knn_model = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=-1)

# Fit mô hình với ma trận Item-User
knn_model.fit(item_user_matrix)

print('✅ Huấn luyện mô hình KNN thành công!')


✅ Huấn luyện mô hình KNN thành công!


## 5. Lưu Mô hình và Dữ liệu (Export to .pkl)
Chúng ta sẽ lưu các đối tượng cần thiết bằng thư viện `joblib` để sử dụng cho Web Demo sau này.


In [6]:
# Lưu các đối tượng vào thư mục models/intermediate/
joblib.dump(knn_model, 'models/intermediate/knn_item_model.pkl')
joblib.dump(song_to_idx, 'models/intermediate/song_to_idx.pkl')
joblib.dump(idx_to_song, 'models/intermediate/idx_to_song.pkl')
joblib.dump(user_profiles, 'models/intermediate/user_profiles.pkl')
joblib.dump(song_details, 'models/intermediate/song_details.pkl')

# Lưu cấu trúc lịch sử người dùng để kiểm tra bài đã nghe
user_history = df_filtered.groupby('msno')['song_id'].apply(set).to_dict()
joblib.dump(user_history, 'models/intermediate/user_history.pkl')

# Tổng hợp thành 1 file duy nhất cho web
knn_final_model = {
    'model': knn_model,
    'song_to_idx': song_to_idx,
    'idx_to_song': idx_to_song,
    'user_profiles': user_profiles,
    'song_details': song_details,
    'user_history': user_history
}
joblib.dump(knn_final_model, 'models/knn_final_model.pkl')

print('💾 Đã lưu các file trung gian vào models/intermediate/ và file tổng hợp vào models/knn_final_model.pkl')


💾 Đã lưu các file trung gian vào models/intermediate/ và file tổng hợp vào models/knn_final_model.pkl


## 6. Xây dựng Hàm Gợi ý Hybrid
Hàm kết hợp:
1. Tìm bài hát mà User đã nghe.
2. Dùng KNN tìm các bài hát tương tự cho mỗi bài đã nghe.
3. Chấm điểm candidate songs:
   - Điểm cơ sở (Base Score): Tần suất xuất hiện trong tập candidate.
   - Boost Score: Cộng thêm điểm nếu bài hát có Genre hoặc Language khớp với `User Taste Profile`.


In [7]:
def recommend_hybrid(user_id, k_neighbors=100, top_n=10, genre_boost=2.0, lang_boost=1.5):
    
    candidate_songs = {}
    
    # TRƯỜNG HỢP 1: User CÓ lịch sử nghe nhạc
    if user_id in user_history and len(user_history[user_id]) > 0:
        
        listened_songs = list(user_history[user_id])
        if len(listened_songs) > 50:
            listened_songs = listened_songs[:50]
            
        for song_id in listened_songs:
            if song_id not in song_to_idx:
                continue
                
            s_idx = song_to_idx[song_id]
            song_vector = item_user_matrix[s_idx]
            distances, indices = knn_model.kneighbors(song_vector, n_neighbors=k_neighbors+1)
            similar_song_indices = indices.flatten()[1:]
            
            for sim_idx in similar_song_indices:
                sim_song_id = idx_to_song[sim_idx]
                if sim_song_id in user_history[user_id]:
                    continue
                candidate_songs[sim_song_id] = candidate_songs.get(sim_song_id, 0) + 1
    
    # TRƯỜNG HỢP 2: User KHÔNG có lịch sử (cold start)
    else:
        if user_id not in user_to_idx_personal:
            popular_songs_list = song_details.head(20).index.tolist()
            for song_id in popular_songs_list:
                candidate_songs[song_id] = candidate_songs.get(song_id, 0) + 1
        else:
            user_idx = user_to_idx_personal[user_id]
            distances, indices = user_knn.kneighbors([X_personal[user_idx]])
            
            similar_users = [idx_to_user_personal[idx] for idx in indices[0][1:6]]
            
            for sim_user in similar_users:
                if sim_user in user_history:
                    for song_id in list(user_history[sim_user])[:30]:
                        candidate_songs[song_id] = candidate_songs.get(song_id, 0) + 1

    if not candidate_songs:
        return "Không có gợi ý phù hợp."

    # Chấm điểm ưu tiên theo User Taste Profile
    user_taste = user_profiles.loc[user_id]
    user_pref_genre = user_taste['top_genre']
    user_pref_lang = user_taste['top_language']
    
    scored_candidates = []
    for song_id, base_score in candidate_songs.items():
        song_info = song_details.loc[song_id]
        final_score = base_score
        
        if song_info['primary_genre'] == user_pref_genre:
            final_score += genre_boost
            
        if song_info['language'] == user_pref_lang:
            final_score += lang_boost
            
        scored_candidates.append((song_id, final_score))
        
    scored_candidates.sort(key=lambda x: x[1], reverse=True)
    top_song_ids = [item[0] for item in scored_candidates[:top_n]]
    
    result_df = song_details.loc[top_song_ids].reset_index()
    result_df['hybrid_score'] = [item[1] for item in scored_candidates[:top_n]]
    
    return result_df[['song_id', 'song_name', 'artist_name', 'primary_genre', 'language', 'hybrid_score']]

## 7. Demo & Kiểm thử Gợi ý
Chạy thử mô hình với một số người dùng ngẫu nhiên.


In [8]:
# Lấy ngẫu nhiên 1 User ID
sample_user = user_profiles.sample(1, random_state=42).index[0]

print(f"👤 Gợi ý cho User: {sample_user}")
print("-" * 50)
print(f"Hồ sơ Sở thích (Taste Profile):")
print(f" - Thể loại yêu thích: {user_profiles.loc[sample_user]['top_genre']}")
print(f" - Ngôn ngữ yêu thích: {user_profiles.loc[sample_user]['top_language']}")

print("\n🎵 Top 10 bài hát gợi ý (Hybrid Item-Based + User Preferences):")
recs = recommend_hybrid(sample_user, top_n=10)
display(recs)


👤 Gợi ý cho User: 3xBCNT7BqgMGbebWUZRtvcEYYKwEbDfWAEAvDrZ5lV8=
--------------------------------------------------
Hồ sơ Sở thích (Taste Profile):
 - Thể loại yêu thích: 465.0
 - Ngôn ngữ yêu thích: 3.0

🎵 Top 10 bài hát gợi ý (Hybrid Item-Based + User Preferences):


,song_id,song_name,artist_name,primary_genre,language,hybrid_score
0,8qWeDv6RTv+hYJxW94e7n6HBzHPGPEZW9FuGhj6pPhQ=,愛．這件事情,傅又宣 (Maggie Fu),465,3.0,10.5
1,OaEbZ6TJ1NePtNUeEgWsvFLeopkSln9WQu8PBR5B3+A=,走著走著就散了,莊心妍,465,3.0,9.5
2,n+pMhj/jpCnpiUcSDl4k3i9FJODDddEXmpE48/HczTI=,修煉愛情 (Practice Love),林俊傑 (JJ Lin),465,3.0,9.5
3,TIwOs7iFTKo3Cy2yiNReYYcZc1JyAx+0k08+z97k1dA=,有點甜,汪蘇瀧 (Silence Wang),465,3.0,9.5
4,+Sm75wnBf/sjm/QMUAFx8N+Ae04kWCXGlgH50tTeM6c=,迷途羔羊,兄弟本色G.U.T.S. (姚中仁、張震嶽、頑童MJ116),465,3.0,8.5
5,GFWjz4apE8zMiXe6wc0qCjr+DbK9BFkwo/rr+FsNm+g=,癡情男子漢,玖壹壹,465,3.0,8.5
6,DLBDZhOoW7zd7GBV99bi92ZXYUS26lzV+jJKbHshP5c=,演員,薛之謙,465,3.0,8.5
7,JA6C0GEK1sSCVbHyqtruH/ARD1NKolYrw7HXy6EVNAc=,愛不需要裝乖 (feat.王詩安) (Love Doesn't Need To Pretend),謝和弦 (R-chord),465,3.0,8.5
8,wBTWuHbjdjxnG1lQcbqnK4FddV24rUhuyrYLd9c/hmk=,小幸運 (A little happiness),田馥甄 (Hebe),465,3.0,8.5
9,q5/Vwifgv0SGdEIyiMU2F0pIXkeCIW3w2xB8n4iMoS0=,打鐵,玖壹壹,465,3.0,8.5


## 8. Đánh giá (Evaluation - Hit Rate@10)
Chúng ta sẽ kiểm tra xem với mô hình mới này, tỷ lệ Hit Rate có tăng lên hay không so với phiên bản User-based thuần túy.


In [9]:
import random

def evaluate_hit_rate_hybrid(test_users, top_n=10):
    hits = 0
    total = 0
    
    for user in test_users:
        user_songs = list(user_history[user])
        if len(user_songs) < 2:
            continue
            
        # Ẩn 1 bài hát
        hidden_song = random.choice(user_songs)
        
        # Sửa lại lịch sử tạm thời
        user_history[user].remove(hidden_song)
        
        # Gợi ý
        recs = recommend_hybrid(user, top_n=top_n)
        
        # Phục hồi lịch sử
        user_history[user].add(hidden_song)
        
        if type(recs) != str and hidden_song in recs['song_id'].values:
            hits += 1
        total += 1
        
    hit_rate = hits / total if total > 0 else 0
    return hit_rate, total

# Lấy 100 users có >= 2 bài hát để test
eligible_users = [u for u, songs in user_history.items() if len(songs) >= 2]
test_users_sample = random.sample(eligible_users, min(100, len(eligible_users)))

print(f"Đang đánh giá Hit Rate@{10} trên {len(test_users_sample)} users...")
hit_rate, total_tested = evaluate_hit_rate_hybrid(test_users_sample, top_n=10)

print(f"✅ Đánh giá hoàn tất!")
print(f"Hit Rate @ 10: {hit_rate:.2%} (Trên tổng {total_tested} test cases)")


Đang đánh giá Hit Rate@10 trên 100 users...
✅ Đánh giá hoàn tất!
Hit Rate @ 10: 21.00% (Trên tổng 100 test cases)
